# LeWM Duckietown Training Only

This notebook runs **training only** (no probe/eval) using the LeWM Hydra trainer in `/content/le-wm`.


In [ ]:
# 1) Training configuration
RUN_NAME = 'duckie_explore_retrain'
DATA_LOCAL = '/content/duckie_explore.h5'
MAX_EPOCHS = 50
BATCH_SIZE = 128
NUM_WORKERS = 2
PRECISION = 'bf16-mixed'

# Source repos / assets
LEWM_REPO = 'https://github.com/lucas-maes/le-wm.git'
DATA_URL = 'https://leworldduckie-public-data.s3.us-east-1.amazonaws.com/duckie_explore.h5'  # public mirror URL

# Optional: set this to a checkpoint if you want resume/init
INIT_CKPT = ''


In [ ]:
# 2) Bootstrap: clone le-wm and fetch data if needed
import os, subprocess

if not os.path.exists('/content/le-wm'):
    subprocess.check_call(['bash','-lc', f'git clone {LEWM_REPO} /content/le-wm'])
else:
    print('/content/le-wm already exists')

if not os.path.exists(DATA_LOCAL):
    if not DATA_URL:
        raise FileNotFoundError(
            f'Missing {DATA_LOCAL}. Set DATA_URL to a presigned URL, then rerun this cell.'
        )
    subprocess.check_call(['bash','-lc', f'wget -O {DATA_LOCAL} "{DATA_URL}"'])
    print('Downloaded dataset to', DATA_LOCAL)
else:
    print('Dataset already present:', DATA_LOCAL)


In [ ]:
# 3) Install dependencies (robust for le-wm repo layout)
import os, subprocess
subprocess.check_call(['bash','-lc','python3 -m pip install -U pip'])
if os.path.exists('/content/le-wm/requirements.txt'):
    cmd = 'cd /content/le-wm && pip install -r requirements.txt'
else:
    # le-wm upstream has no requirements.txt; install known training stack
    cmd = 'pip install stable-worldmodel[train,env] h5py einops matplotlib scikit-learn xvfbwrapper boto3'
print('Running:', cmd)
subprocess.check_call(['bash','-lc', cmd])
print('Dependencies installed.')


In [ ]:
# 4) Environment checks
import os, subprocess
print('python:', subprocess.check_output(['python3','--version'], text=True).strip())
print('data exists:', os.path.exists(DATA_LOCAL), DATA_LOCAL)
if not os.path.exists('/content/le-wm/train.py'):
    raise FileNotFoundError('Missing /content/le-wm/train.py even after bootstrap.')


In [ ]:
# 5) Launch training (writes log to /content/train_duckie.log)
import subprocess

overrides = [
    f'data=duckietown',
    f'subdir={RUN_NAME}',
    f'data.path={DATA_LOCAL}',
    f'trainer.max_epochs={MAX_EPOCHS}',
    f'loader.batch_size={BATCH_SIZE}',
    f'loader.num_workers={NUM_WORKERS}',
    f'trainer.precision={PRECISION}',
    'wandb.enabled=false',
]
if INIT_CKPT:
    overrides.append(f'checkpoint.path={INIT_CKPT}')

cmd = 'cd /content/le-wm && HYDRA_FULL_ERROR=1 python3 train.py ' + ' '.join(overrides) + ' 2>&1 | tee /content/train_duckie.log'
print(cmd)
ret = subprocess.run(['bash','-lc',cmd])
if ret.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {ret.returncode}. See /content/train_duckie.log')
print('Training finished.')


In [ ]:
# 6) Locate checkpoints
import glob, os
cands = sorted(glob.glob(f'/content/stable-wm/{RUN_NAME}/*.ckpt'))
if not cands:
    print('No checkpoints found under', f'/content/stable-wm/{RUN_NAME}')
else:
    for p in cands[-10:]:
        print(p, os.path.getsize(p))


In [ ]:
# 7) Optional live monitor (run in separate cell while training is active)
# !tail -f /content/train_duckie.log
# !watch -n 30 'ls -lh /content/stable-wm/duckie_explore_retrain/*.ckpt 2>/dev/null || echo no-ckpt-yet'
